TF-IDF

In [9]:
from google.colab import files
uploaded = files.upload()

Saving final_ground_truth_dataset.csv to final_ground_truth_dataset.csv


In [10]:

import pandas as pd

df = pd.read_csv("final_ground_truth_dataset.csv")

df.head()


,reviewId,userName,content,rating,thumbsUpCount,replyContent,rated_app_version,annotator_1_sentiment,vader_sentiment,vader_compound,afinn_score,afinn_sentiment,ground_truth_sentiment
0,1,Leo Diaz,it doesn't keep your stuff when you delete and...,1,0,thank you for sharing your experience. losing ...,unknown,Negative,Negative,-0.5719,-3.0,Negative,Negative
1,2,Ryan Klutts,fun and play offline,5,0,NaN,3.60.0,Positive,Positive,0.6369,3.0,Positive,Positive
2,3,Ellen Yeboah,cool,5,0,NaN,unknown,Positive,Positive,0.3182,1.0,Positive,Positive
3,6,Ahmadabalrahman Ahmad,it's good,5,0,NaN,3.60.0,Positive,Positive,0.4404,3.0,Positive,Positive
4,9,Ramlat Kabir Wada,i love this game it deserves 5 stars and works...,5,0,NaN,unknown,Positive,Positive,0.6369,3.0,Positive,Positive


In [11]:

#  selects the review text column that will be converted into numeric vectors.

texts = df["content"].astype(str)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)


In [13]:

tfidf_matrix = tfidf_vectorizer.fit_transform(texts)

In [14]:

# This cell converts the TF-IDF matrix into a DataFrame.
# Each column represents a word feature.

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

tfidf_df.head()


,10,105,14,2016,2017,able,absolute,absolutely,access,account,...,wait,want,watch,watching,wifi,wo,words,wordy,works,world
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.383712,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.420431,0.0


In [15]:
tfidf_df.to_csv("tfidf_features.csv", index=False)

TF-IDF + SVM

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [17]:
# Load TF-IDF feature matrix
X = pd.read_csv("tfidf_features.csv")
print("TF-IDF feature shape:", X.shape)

TF-IDF feature shape: (63, 250)


In [18]:
# Load dataset that contains ground truth labels
df_labels = pd.read_csv("final_ground_truth_dataset.csv")
y = df_labels["ground_truth_sentiment"]
print(y.value_counts())


ground_truth_sentiment
Positive    48
Neutral     10
Negative     5
Name: count, dtype: int64


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
svm_model = LinearSVC()

In [21]:
svm_model.fit(X_train, y_train)

LinearSVC()

In [22]:
y_pred = svm_model.predict(X_test)

In [23]:

print("Accuracy:", accuracy_score(y_test, y_pred))
#print("\nClassification Report:\n")
#print(classification_report(y_test, y_pred))


Accuracy: 0.7692307692307693


GLOVE

In [24]:
!pip install gensim

In [25]:
import pandas as pd
import numpy as np
import gensim.downloader as api

In [26]:
# Load dataset
df = pd.read_csv("final_ground_truth_dataset.csv")
texts = df["content"].astype(str)

In [27]:
#pretrained glove
glove_model = api.load("glove-wiki-gigaword-100")

[==================================================] 100.0% 128.1/128.1MB downloaded


In [28]:
def text_to_glove_vector(text, model, dim=100):
    words = text.lower().split()
    vectors = [model[word] for word in words if word in model]
    if len(vectors) == 0:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

    # Convert all reviews to GloVe vectors
X_glove = texts.apply(lambda x: text_to_glove_vector(x, glove_model))
X_glove = pd.DataFrame(X_glove.tolist())

In [29]:
# Save features
X_glove.to_csv("glove_features.csv", index=False)

GOLVE + LINEAR REFRESSION

In [30]:
#golve + linear regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score

In [32]:
# Load GloVe feature matrix
X = pd.read_csv("glove_features.csv")

print("GloVe feature shape:", X.shape)

GloVe feature shape: (63, 100)


In [31]:
df = pd.read_csv("final_ground_truth_dataset.csv")

# Map sentiment labels to numerical values
label_mapping = {
    "Negative": -1,
    "Neutral": 0,
    "Positive": 1
}

y = df["ground_truth_sentiment"].map(label_mapping)

y.value_counts()



,count
ground_truth_sentiment,
1,48
0,10
-1,5


In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [34]:
# Initialize Linear Regression
lin_reg = LinearRegression()

# Train
lin_reg.fit(X_train, y_train)

LinearRegression()

In [35]:
# Predict continuous sentiment scores
y_pred_scores = lin_reg.predict(X_test)

y_pred_scores[:10]

array([ 0.88732083,  0.74426174,  0.30177206,  0.76076126,  1.25032977,
       -0.67322181,  0.1243231 ,  0.93650253,  1.18899347, -1.36427143])

In [36]:
def score_to_label(score):
    if score > 0.5:
        return "Positive"
    elif score < -0.5:
        return "Negative"
    else:
        return "Neutral"

In [37]:
# Apply thresholding
y_pred_labels = [score_to_label(score) for score in y_pred_scores]

# Convert true labels back to strings
y_true_labels = y_test.map({-1: "Negative", 0: "Neutral", 1: "Positive"})

In [38]:
print("Accuracy:", accuracy_score(y_true_labels, y_pred_labels))
print("\nClassification Report:\n")
print(classification_report(y_true_labels, y_pred_labels))

Accuracy: 0.6923076923076923

Classification Report:

              precision    recall  f1-score   support

    Negative       1.00      0.67      0.80         3
     Neutral       0.25      0.50      0.33         2
    Positive       0.86      0.75      0.80         8

    accuracy                           0.69        13
   macro avg       0.70      0.64      0.64        13
weighted avg       0.80      0.69      0.73        13

